<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/CIBIL_loan_allotment_prediction_credit_risk_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# 1. Load Data
df = pd.read_csv("india_personal_loan_default_risk_2026.csv")
print(df.columns)

# 2. Feature Engineering
df['dti_ratio'] = df['existing_emi_monthly_inr'] / (df['monthly_income_inr'] + 1)
df['projected_emi'] = df['loan_amount_requested_inr'] / (df['loan_tenure_months'] + 1e-6) # Added small epsilon to prevent division by zero
df['total_dti_ratio'] = (df['existing_emi_monthly_inr'] + df['projected_emi']) / (df['monthly_income_inr'] + 1)

# Categorize CIBIL into standard Indian credit bands
df['cibil_band'] = pd.cut(
    df['cibil_score'],
    bins=[0, 600, 700, 750, 900],
    labels=['Poor', 'Fair', 'Good', 'Excellent']
)

# 3. Categorical Handling
cat_cols = [
    'gender', 'marital_status', 'city_tier', 'state',
    'employment_type', 'loan_purpose', 'collateral_provided', 'cibil_band'
]
for col in cat_cols:
    df[col] = df[col].astype('category')

# Define target and predictors (exclude ID and target proxy metrics)
drop_cols = ['applicant_id', 'default_flag', 'default_risk_score']
features = [c for c in df.columns if c not in drop_cols]

X = df[features]
y = df['default_flag']

# 4. Stratified Train-Test Split (6% target imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Model Training with Imbalance Compensation
# Calculate class balance ratio for scale_pos_weight
neg_pos_ratio = (len(y_train) - sum(y_train)) / sum(y_train)

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=31,
    scale_pos_weight=neg_pos_ratio,
    random_state=42
)

model.fit(X_train, y_train)

# 6. Prediction & Evaluation
y_probs = model.predict_proba(X_test)[:, 1]

# Adjust threshold based on business risk tolerance (e.g., 0.50 default threshold)
y_pred = (y_probs >= 0.50).astype(int)

print("=== Performance Metrics ===")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}")
print(f"PR-AUC (Average Precision): {average_precision_score(y_test, y_probs):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 7. Pricing Risk Audit
test_audit = X_test.copy()
test_audit['actual_default'] = y_test
test_audit['predicted_risk'] = y_probs
test_audit['interest_rate_offered'] = df.loc[X_test.index, 'interest_rate_offered_pct']

# Identify mispriced loans (High risk > 40% prob, but offered sub-16% interest rate)
mispriced_loans = test_audit[(test_audit['predicted_risk'] > 0.40) & (test_audit['interest_rate_offered'] < 16.0)]
print(f"\nUnder-priced high-risk loans detected: {len(mispriced_loans)}")

Index(['applicant_id', 'age', 'gender', 'marital_status', 'city_tier', 'state',
       'employment_type', 'bank_account_vintage_years', 'monthly_income_inr',
       'existing_loans_count', 'existing_emi_monthly_inr',
       'credit_utilization_ratio_pct', 'num_credit_inquiries_last_6m',
       'late_payments_last_12m', 'loan_amount_requested_inr',
       'loan_tenure_months', 'loan_purpose', 'collateral_provided',
       'cibil_score', 'interest_rate_offered_pct', 'default_risk_score',
       'default_flag'],
      dtype='object')
[LightGBM] [Info] Number of positive: 1228, number of negative: 18772
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003085 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2637
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
from sklearn.model_selection import train_test_split

# 1. DATA PREPARATION & FEATURE ENGINEERING
df = pd.read_csv("india_personal_loan_default_risk_2026.csv")

df['dti_ratio'] = df['existing_emi_monthly_inr'] / (df['monthly_income_inr'] + 1)
df['projected_emi'] = df['loan_amount_requested_inr'] / df['loan_tenure_months']
df['total_dti_ratio'] = (df['existing_emi_monthly_inr'] + df['projected_emi']) / (df['monthly_income_inr'] + 1)

df['cibil_band'] = pd.cut(
    df['cibil_score'],
    bins=[0, 600, 700, 750, 900],
    labels=['Poor', 'Fair', 'Good', 'Excellent']
)

cat_cols = ['gender', 'marital_status', 'city_tier', 'state', 'employment_type', 'loan_purpose', 'collateral_provided', 'cibil_band']
for col in cat_cols:
    df[col] = df[col].astype('category')

drop_cols = ['applicant_id', 'default_flag', 'default_risk_score']
feature_names = [c for c in df.columns if c not in drop_cols]

X = df[feature_names]
y = df['default_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 2. MODEL TRAINING
neg_pos_ratio = (len(y_train) - sum(y_train)) / sum(y_train)
model = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.03, num_leaves=31,
    scale_pos_weight=neg_pos_ratio, random_state=42
)
model.fit(X_train, y_train)
y_probs = model.predict_proba(X_test)[:, 1]

# 3. ASYMMETRIC COST-SENSITIVE THRESHOLD OPTIMIZATION
def optimize_threshold(y_true, y_probs, avg_principal=200000, lgd=0.70, profit_margin=0.10):
    """
    LGD (Loss Given Default): Bank loses 70% of principal on false negative.
    Opportunity Cost: Bank loses 10% expected interest profit on false positive.
    """
    thresholds = np.linspace(0.01, 0.90, 100)
    best_thresh, min_loss = 0.50, float('inf')

    for t in thresholds:
        preds = (y_probs >= t).astype(int)
        fn = np.sum((preds == 0) & (y_true == 1)) # Approved defaulted borrower
        fp = np.sum((preds == 1) & (y_true == 0)) # Rejected solvent borrower

        total_loss = (fn * avg_principal * lgd) + (fp * avg_principal * profit_margin)
        if total_loss < min_loss:
            min_loss = total_loss
            best_thresh = t

    return best_thresh, min_loss

optimal_threshold, min_portfolio_loss = optimize_threshold(y_test, y_probs)
print(f"Optimal Decision Cutoff Threshold: {optimal_threshold:.4f}")

# 4. CITY TIER & DEMOGRAPHIC FAIRNESS AUDIT
def audit_city_tier_fairness(data_df):
    audit_summary = data_df.groupby('city_tier', observed=False).agg(
        Total_Applicants=('applicant_id', 'count'),
        Default_Rate=('default_flag', 'mean'),
        Avg_CIBIL=('cibil_score', 'mean'),
        Avg_Monthly_Income=('monthly_income_inr', 'mean')
    ).reset_index()
    return audit_summary

print("\n--- Fairness Audit across City Tiers ---")
print(audit_city_tier_fairness(df))

# 5. SHAP EXPLAINER INITIALIZATION
explainer = shap.TreeExplainer(model)

# 6. SINGLE APPLICANT DECISION & PRICING ENGINE
def evaluate_loan_applicant(applicant_dict, model, threshold, explainer, feature_names):
    """
    Evaluates a single borrower profile and generates an underwriting verdict.
    """
    input_df = pd.DataFrame([applicant_dict])

    # Feature Engineering
    input_df['dti_ratio'] = input_df['existing_emi_monthly_inr'] / (input_df['monthly_income_inr'] + 1)
    input_df['projected_emi'] = input_df['loan_amount_requested_inr'] / input_df['loan_tenure_months']
    input_df['total_dti_ratio'] = (input_df['existing_emi_monthly_inr'] + input_df['projected_emi']) / (input_df['monthly_income_inr'] + 1)

    input_df['cibil_band'] = pd.cut(
        input_df['cibil_score'],
        bins=[0, 600, 700, 750, 900],
        labels=['Poor', 'Fair', 'Good', 'Excellent']
    )

    for col in cat_cols:
        input_df[col] = input_df[col].astype('category')

    X_single = input_df[feature_names]
    prob_default = model.predict_proba(X_single)[0, 1]

    # Compute SHAP Values for local explanation
    shap_vals = explainer(X_single).values[0]
    if len(shap_vals.shape) > 1:
        shap_vals = shap_vals[:, 1]

    # Isolate top features pushing risk upward
    risk_factors = sorted(zip(feature_names, shap_vals), key=lambda x: x[1], reverse=True)
    top_adverse_reasons = [f"{feat} (SHAP contribution: +{val:.3f})" for feat, val in risk_factors[:3] if val > 0]

    # Risk-Based Pricing Matrix & Decision Rule
    if prob_default >= threshold:
        status = "DECLINED"
        offered_rate = "N/A"
        reasons = top_adverse_reasons
    else:
        status = "APPROVED"
        reasons = "Applicant meets risk threshold."
        if prob_default < 0.02:
            offered_rate = "11.5% (Prime)"
        elif prob_default < 0.05:
            offered_rate = "14.0% (Standard)"
        else:
            offered_rate = "17.5% (Subprime / Collateral Required)"

    return {
        "Underwriting Decision": status,
        "Default Probability": f"{prob_default:.2%}",
        "Assigned Risk Cutoff": f"{threshold:.2%}",
        "Offered Interest Rate": offered_rate,
        "Adverse Action Reasons": reasons
    }

# TEST SINGLE APPLICANT INFERENCE
sample_applicant = {
    "age": 32,
    "gender": "Male",
    "marital_status": "Married",
    "city_tier": "Tier 3",
    "state": "Bihar",
    "employment_type": "Self-Employed Professional",
    "bank_account_vintage_years": 1.2,
    "monthly_income_inr": 22000,
    "existing_loans_count": 3,
    "existing_emi_monthly_inr": 12000, # Changed from 'existing_emi_inr'
    "credit_utilization_ratio_pct": 68.5,
    "num_credit_inquiries_last_6m": 4,
    "late_payments_last_12m": 2,
    "loan_amount_requested_inr": 250000,
    "loan_tenure_months": 36,
    "loan_purpose": "Personal Expense",
    "collateral_provided": "No",
    "cibil_score": 580,
    "interest_rate_offered_pct": 22.0,
    "default_risk_score": 35.0
}

decision_output = evaluate_loan_applicant(sample_applicant, model, optimal_threshold, explainer, feature_names)
print("\n--- Live Single Applicant Underwriting Result ---")
for k, v in decision_output.items():
    print(f"{k}: {v}")

[LightGBM] [Info] Number of positive: 1228, number of negative: 18772
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001711 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2637
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.061400 -> initscore=-2.726980
[LightGBM] [Info] Start training from score -2.726980
Optimal Decision Cutoff Threshold: 0.5494

--- Fairness Audit across City Tiers ---
  city_tier  Total_Applicants  Default_Rate   Avg_CIBIL  Avg_Monthly_Income
0    Tier 1              9978      0.056324  615.235318        68924.113049
1    Tier 2              9501      0.065151  615.107778        50595.947795
2    Tier 3              5521      0.064119  615.705307        36655.279841

--- Live Single Applicant Underwriting Result ---
Unde

Making an interactive site


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd

st.set_page_config(page_title="Indian Lending Risk Engine", layout="wide")
st.title("Credit Risk & Underwriting Decision System")

col1, col2, col3 = st.columns(3)

with col1:
    st.subheader("Borrower Profile")
    monthly_income = st.number_input("Monthly Income (INR)", min_value=10000, value=50000, step=5000)
    cibil_score = st.slider("CIBIL Score", 300, 900, 680)
    city_tier = st.selectbox("City Tier", ["Tier 1", "Tier 2", "Tier 3"])
    employment_type = st.selectbox("Employment Type", ["Salaried - Private", "Salaried - PSU", "Salaried - Government", "Self-Employed Professional", "Business Owner"])

with col2:
    st.subheader("Existing Liabilities")
    existing_emi = st.number_input("Existing Monthly EMIs (INR)", min_value=0, value=10000, step=1000)
    utilization = st.slider("Credit Utilization (%)", 0.0, 100.0, 35.0)
    late_payments = st.selectbox("Late Payments (Last 12M)", [0, 1, 2, 3, 4, 5])
    inquiries = st.selectbox("Credit Inquiries (Last 6M)", [0, 1, 2, 3, 4, 5])

with col3:
    st.subheader("Loan Request")
    loan_amount = st.number_input("Requested Loan Amount (INR)", min_value=10000, value=200000, step=10000)
    loan_tenure = st.selectbox("Tenure (Months)", [12, 24, 36, 48, 60])
    collateral = st.radio("Collateral Provided?", ["No", "Yes"])

if st.button("Evaluate Application"):
    projected_emi = loan_amount / loan_tenure
    total_dti = (existing_emi + projected_emi) / (monthly_income + 1)

    st.markdown("---")
    st.subheader("Decision Summary")

    if cibil_score < 620 or total_dti > 0.55 or late_payments >= 2:
        st.error("Underwriting Verdict: DECLINED")
        st.write(f"**Calculated Total DTI:** {total_dti:.1%}")
        st.write("**Top Adverse Reasons:**")
        if cibil_score < 620: st.write("- CIBIL score below risk tolerance cutoff.")
        if total_dti > 0.55: st.write("- Total Debt-To-Income exceeds max allowable threshold (55%).")
        if late_payments >= 2: st.write("- High recent delinquency count in past 12 months.")
    else:
        st.success("Underwriting Verdict: APPROVED")
        st.write(f"**Calculated Total DTI:** {total_dti:.1%}")
        if cibil_score >= 750:
            st.metric("Recommended Interest Rate", "11.5% (Prime)")
        elif cibil_score >= 680:
            st.metric("Recommended Interest Rate", "14.2% (Standard)")
        else:
            st.metric("Recommended Interest Rate", "17.0% (Risk-Adjusted)")

Overwriting app.py


In [ ]:
!pip install streamlit shap lightgbm pandas -q

In [ ]:
!curl ipv4.icanhazip.com

34.26.202.102


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸

⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-08-30 07:05:22.879 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.26.202.102:8501

  Stopping...
^C


In [ ]:
!streamlit run app.py & npx -y localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙2026-08-30 07:08:23.004 Uvicorn server started on :::8501
⠹⠸
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.26.202.102:8501

⠼⠴⠦⠧⠇your url is: https://silly-bats-send.loca.lt
